# Scalable image compression with embedded zero tree coding
In this [tutorial](./image-coding-wavelet.ipynb) we discussed about the spatial scalability feature intrinsically offered by Wavelet transforms. Such a property allows to decode a given bitstream up to a desired spatial resolution, hence limiting transmission bandwidth and/or decoder's complexity. We also mentioned that spatial scalability can be combined with quality scalability so that not only the target resolution is attained but also a gracefully degraded version of the compressed image can be achieved. These two scalability dimensions are desirable in different applications scenarios whereby an image/video needs to be browsed quickly as preview (e.g. in media assets archiving applications) or network connections are not reliable enough to guarantee full transmission of the whole compressed bitstream. In this tutorial we will explore quality scalability by extending the Simple Wavelet Image Coding (SWIC) scheme and using the well-known Set Partitioning In Hierarchical Trees ([SPIHT](https://www.researchgate.net/publication/2835826_A_New_Fast_and_Efficient_Image_Codec_Based_on_Set_Partitioning_in_Hierarchical_Trees)) method.

In the following, the concept of embedded bitstream representation is recalled to highlight its importance in quality scalability. A quick summary on the main methods developed to achieve quality scalability in Wavelet-based compression is then provided, with the Embedded Zerotree Wavelet (EZW) and Embedded Block Coding with Optimised Truncation (EBCOT) approaches reviewed. The method considered in this tutorial (SPIHT), is then described in a greater level of detail next. Its implementation in the SWIC codec follows, along with its performance over real images. Quality scalability has been a very active topic of research at the end of the nineties, early 2000s and plenty of good references are available to study this subject. Two good references are the following textbooks:
 * David S. Taubman and Micheal W. Marcellin, "JPEG 2000: Image compression fundamentals, standards and practice", Kluwer Academic Press, 773 pages, 2002.
 * Khalid Sayood, "Introduction to data compression", Morgan Kaufmann Publisher, 680 pages, 3rd edition, 2006.

## Quality scalability and embedded bitstreams
Quality scalability refers to the ability of an image/video coding system to compress the original content to a distortion level $D$ so that a progressive parsing (decoding) of the compressed bitstream will produce a progressively decreasing distortion, ultimately equating $D$ in case of full decoding. To enable such a progressive quality refinement, the compression system needs to produce a so-called embedded bitstream. When a bitstream is embedded, the receiver can stop its decoding after a given number of bits ($B$) is received and yet the whole image is intelligible (but has visual distortion larger than $D$). Such a definition might appear sort of tautological, so consider a counter example of a bitstream **which is not embedded**: the one specified by the [Baseline profile of the JPEG standard](../jpeg/jpeg-baseline.ipynb). There, after decoding $B$ bits, the reconstructed image might appear as the one portrayed in the following.

<img src="non-embedded-example.png" alt="Example of non embedded bitstream when it is not fully decoded" width="250"/>

As we may see, only a subset of all image's pixels is reconstructed to the encoding distortion ($D$) whilst the remaining pixels are not decoded at all, since their associated bits go beyond received bits ($B$). This example of *non embedded* bitstream should hint at the main principle behind embedded image and video compression: organise/transmit compressed data differently. For the JPEG Baseline profile, one could think to transmit first all DC transform coefficients so that a spatial resolution equal to one eigth of the original image can be achieved (recall that JPEG Baseline uses $8\times8$ blocks as units for compression and the DC coefficient represents the average of all pixels in each of these blocks). Following on this approach, subsequent AC coefficients can be transmitted so that the quality of the decoded image can improve gradually as more bits are received and decoded. The order at which AC coefficients are transmitted may be the zigzag one, since it resembles a descending order of each coefficient's magnitude. As such, it should be clear at this point one of the key aspects in the design of embedded bitstreams: transmit first those data which lead to the largest distortion reduction. Thankfully, spatial transformations such as the DCT or Wavelet provide by design such a ranking in their transform coefficients.

Another aspect to consider when designing embedded compression systems, is the level of granularity at which quality is recovered. In the example made on how to make a JPEG Baseline's bitstream embedded, it is easy to understand that if the DC coefficients require $B_{DC}$ bits and less bits are transmitted, then the decoded image will still have the patchy aspect as portrayed earlier. Accordingly, in this particular example, bits should be transmitted in bursts of length of (at least) $B_{DC}$. Given that in some applications it would be desirable to have a finer control of bits transmitted, data in an embedded bitstream should be written not only with respect to their coefficient class but also according to their binary representation, starting from their sign and Most Significant Bit (MSB), to then proceed with all remaing less significant bits. In this way, fine granularity (often also denoted as fine grain) quality scalability is achieved. In fact, it is well-known that most of the information conveyed by transform coefficients is primarily concentrated in their first MSBs. Considering the binary representation of all transform coefficients, all bits associated with a given position in this representation form a **bitplane**. There will then be the bitplane for MSBs, followed by the one for position MSB - 1, etc. Transmitting these bitplanes starting from the MSBs one is often denoted as *bitplane encoding*. It is a very convenient way to design embedded compression systems (producing embedded bitstreams) which allows for fine grain quality scalability.

## Bitplane encoding methods based on embedded zero trees
Bitplane encoding was already considered in the extensions of the JPEG compression standard (see Annex G of [ITU-T T.81](https://www.itu.int/rec/T-REC-T.81)). Besides allowing for fine quality scalability, bitplane encoding should also be designed to minimise the overall coding rate. As an example, bits within a given biplane may be scanned according to a particular pattern so that long sequences of zeros can be efficiently encoded using run length encoding. For the Wavelet transform, which is the frequency transformation used in the SWIC codec, the concept of *zero trees* is employed. In subband transforms such as the Wavelet there is an intrinsic parent-child relationship between transform coefficients in a given subband at decomposition level $l$ and their counterpart at level $l-1$. More precisely, if a subband transform coefficient has small magnitude at a given level, it will have even smaller values for lower levels of the dyadic decomposition operated by the Wavelet. Consider the following figure which depicts an example of parent-child relationship associated with the conventional Mallat's Wavelet decomposition.

<img src="zero-tree-relationship.png" alt="Example of parent-child relationship for Wavelet decomposition" width="250"/>

It may be noted that each transform coefficients belonging to subband $LL_2$ has three children whilst each transform coefficient in subbands $HL_2$, $LH_2$ and $HH_2$ will have four children. Following on the descendants relationship portrayed, each coefficient in these subbands will also have sixteen grandchildren. As mentioned earlier, if for example a coefficient in the $HL_2$ subband has all bitplanes $b_{K}b_{K-1}\ldots b_{k}$ equal to zero, then its descendants in $HL_1$ and $HL_0$ will also likely have the same bitplanes equal to zero. Considering coefficients in subbands associated with the highest level of Wavelet decomposition as roots and all descendants as nodes of the parent-child relationship, the structure formed is a tree like one, which is referred as *zero tree*: if the root is zero, then all descending nodes will be zero too. Intuitively, we notice that zero trees offer a compact way to encode with a few bits the fact that a bitplane is zero or not across all its associated coefficients. The concept of zero tree was originally introduced by [Jerome Shapiro in 1993](https://web.stanford.edu/class/ee398a/handouts/papers/Shapiro%20-%20EZT.pdf) with his seminal paper on the Embedded Zerotree Wavelet algorithm (EZW). Based on the EZW algorithm, there have been different improvements for embedded quality scalability Wavelet coding where the most notable ones are the Set Partitioning Hierarchical Trees (SPIHT) and, albeit not strictly based on zero trees, the Embedded Block Coding with Optimised Truncation (EBCOT). Since the SPIHT algorithm is the topic of this tutorial, the following two subsections will briefly summarise the main principles behing EZW and EBCOT, leaving a more detailed description of SPIHT to the next sections.

### Embedded Zero tree Wavelet algorithm (EZW)
In EZW the input image is pre-processed by subtracting its average sample mean to all pixels before applying Wavelet decomposition. In this way each subband will have zero mean value. Then a set of thresholds $T_k$ with $k = 0, 1, \ldots, K-1$ is considered whereby $T_k = T_0\cdot 2^{-k}$ and:
$$
\frac{\max{|c|}}{2} < T_0 \le \max{|c|},
$$

having $\max{|c|}$ as the maximum absolute value across all transform coefficients. In EZW a coefficient is said to be significant with respect to $T_k$ if $|c| \geq T_k$. Let's rewrite this condition as:

$$
\frac{|c|}{T_{K-1}} \geq \frac{T_k}{T_{K-1}}.
$$
By taking the nearest integer lower than the division's result on the left hand side (i.e. the floor operator $\lfloor\cdot\rfloor$), we notice that this integer division corresponds to a scalar quantisation operation with step $\Delta \equiv T_{K-1}$. Therefore checking whether $c$ is significant with respect to $T_k$, translates into checking whether the magnitude of its quantisation index $q$ is greater than or equal to $2^{K - 1 - k}$. This consideration is rather important since we can now determine whether $c$ is significant for $T_k$ by verifying that at least one of the bins: $b_0b_1\ldots$ associated with the binary representation of $q$ is equal to 1. Accordingly, EZW organises all image's transform coefficients as $K+1$ binary arrays having the same image's height and width, and each array representing bitplane $b_k$ in the binary representation of each index $q$. The additional array (+1 above) is associated with the sign of each $q$. These binary arrays are then scanned in raster scan order, starting from the Wavelet's subbands associated with the highest decomposition level. Two passes are then performed:

- **Dominant pass**: For all insignificant coefficients (i.e. where none of the bits in the current bitplane array is set to 1) four possible symbols are coded as follows:
  - *POS*: The coefficient becomes significant in the current bitplane and has sign positive
  - *NEG*: The coefficient becomes significant in the current bitplane and has sign negative
  - *ZTR*: The coefficient is insignificant in this bitplane and all its descendants are not significant
  - *IZ*: The coefficient is insignificant in this bitplane but at least one of its descendants is significant
- **Subordinate pass**: For all coefficients which are already marked as significant when coding previous bitplanes, a bit is sent corresponding to the bit's value of this coefficients in the current bitplane

To further improve the coding efficiency of the EZW method, symbols *POS*, *NEG*, *ZTR*, *IZ*, 0 and 1 are encoded using an M-ary context-based arithmetic encoding with five contexts: four used in the dominant pass and one for the subordinate pass.

### Embedded Block Coding with Optimised Truncation (EBCOT)
The EBCOT method was originally proposed in 1994 in Prof. David Taubman's Ph.D. dissertation *"Directionality and Scalability in Image and Video Compression"* and it has been extensively studied during the standardisation activities for the JPEG 2000 standard, ending up to be adopted as the entropy encoding method.

As the name eloquentely explains, EBCOT partitions each subband into a non overlapping grid of codeblocks, each of size $M\times N$ unless the the block falls on any of the subband's boundaries. Within each codeblock, bitplane encoding is applied by repeating the following ordered three steps:
- **Significance propagation**: All non significant transform coefficients are analysed with respect to their eight-sample neighbourhood to understand which coefficients will become significant at a particular bitplane $b$
- **Magnitude refinement**: All significant coefficients are considered and their bits encoded to gradually provide increasing scalable quality
- **Cleanup**: This pass considers again the non significant transform coefficients and encodes them using also run length techniques to improve the coding efficiency

The attractive feature of EBCOT is that each codeblock is independently decodable, hence parallel encoding can be performed both in hardware and software implementations. We notice that the same principle of independent decodability is also used in the SWIC codec. Rather than considering bitplanes, EBCOT introduces the concept of *quality layer*. Each quality layer represents a set of embedded codeblocks bitstreams which minimise the coding distortion under a given rate (bitstream length) constraint. Aside from the first quality layer denoted as $Q_0$, subsequent layers $Q_l$ have incremental lengths $L^l - L^{l-1}$ derived still by minimising the coding distortion $D^l$. To signal whether a codeblock contributes to a given quality layer, its length, etc. EBCOT uses the concept of *tagtree* to efficiently exploit sample redundancy within each codeblock.


## The Set Partitioning In Hierarchical Trees (SPIHT) algorithm
The SPIHT algorithm is based on the main principle discussed earlier to achieve embedded coding: coded data should be sorted according to their decreasing magnitude value. Such sorting guarantees a progressive quality improvement as more bits are parsed by the decoder. The following figure provides a convenient way to organise the image's transform coefficients, considered for bitplane encoding.

<img src="coeffs-bits-sorting.png" alt="Example of sorting for transform coefficients and associated bitplanes" width="500"/>

As we may notice, for each bitplane ($n$) there is only a given number of coefficients which start to become significant. The encoder can therefore send quantity $\mu_n$, corresponding to the number of transform coefficients whose absolute value ($|c_{i,j}|$) is: $2^n \leq |c_{i,j}| < 2^{n+1}$, along with the sign bits for coefficients $c_{i,j}$ and the bits associated with coefficients which became significant at bitplane $m$ with $m > n$. Sign bits in the figure above are denoted by $s$. The encoding procedure can be described by the following pseudo code.

1. Set $n = \lfloor \log_2(\max_{i,j}\{|c_{i,j}|\})\rfloor$ and transmit it to the receiver
1. Transmit $\mu_n$ along with the sign bits and coordinates of those transform coefficients $c_{i,j}$ such that $2^n \leq |c_{i,j}| < 2^{n+1}$
1. Transmit the $n$-th bit of all coefficients such that $|c_{i,j}| \geq 2^{n+1}$
1. Set $n = n - 1$ and go to Step 2

In the paper proposing the SPIHT algorithm, Step 2 and 3 of the pseudo code above are denoted as: ***sorting*** and ***refinement*** pass, respectively. The encoding procedure outlined in this pseudo code suffers by a main drawback: a large share of the coding rate is spent in Step 2 by signalling both $\mu$ and coordinates information.

### A better sorting algorithm: Set partitioning
To save on the large share of coding bits associated with $\mu_n$ and the coefficients' coordinates information in the sorting pass, we need to find a way to avoid transmitting these bits. We start by observing that if the sorting algorithm is shared between the encoder and decoder, this latter can recover the whole execution path by simply knowing the results of comparisons operated by the encoder at any of the algorithm branching points. More specifically, the decoder will then be able to recover $\mu_n$ and the coefficients' coordinates information by the results of comparisons and the execution path.

Next, we also observe that the sorting procedure described in the pseudo code above can be viewed as a selection of those coefficients whose magnitude is $2^n \leq |c_{i,j}| < 2^{n+1}$. In the SPIHT algorithm's terminology (but also generally in embedded scalable coding) these coefficients are said to be significant. The authors of SPIHT propose to partition the set of all coefficients into *partitioning subsets* $T_m$ where the significance test is run and $T_m$ is labelled as *significant* if any of its coefficients has magnitude greater or equal to $2^n$. If none of the coefficients meets that condition, then $T_m$ is *insignificant*. Such a set partitioning approach has the potential to reduce the number of coding bits used by the sorting procedure given that now a significance message (bit) is communicated to the decoder for potentially a large number of coefficients. The association between significance test and bit transmitted can be defined as follows:

$$
S_n(T) = \left\{
    \begin{array}{cl}
        1, & \max\limits_{(i, j)\in T}\{|c_{i,j}|\}\geq 2^n\\
        0, & \text{otherwise}
\end{array}\right.
$$

In case of subsets containing only a single coefficient, the notation above is further simplified as: $S_n(i,j)$. When a subset is significant, it is partitioned according to a rule which is shared between the encoder and decoder and the process continues until all coefficients and values for $n$ have been considered. To reduce even more the number of coding bits, the partitioning should be performed so that subsets expected to be insignificant will contain large number of coefficients whilst significant subsets will have a single coefficient. The implicit hierarchy associated with the Wavelet transform decomposition can be used to this purposes as explained in the next subsection.

### Efficient use of the Wavelet decomposition pyramid: Spatially oriented trees

The subband decomposition operated by the Wavelet transform allows to concentrate the image's energy in the higher levels of the pyramid associated with such a decomposition. Moreover, there is a self similarity between subbands at different levels of the pyramid. As such, if a coefficient is insignificant for a given $n$ in a particular level, it is expected to remain insignificant at lower levels. SPIHT exploits these two properties by defining a tree structure called *spatial orientation trees* where each node is identified by its coefficient's coordinates and can have either four offspring or none (i.e. it is a leaf node). Offspring are always grouped into blocks of $2\times 2$ coefficients. The following picture depicts a spatial orientation tree for a subband pyramid constituted by two levels of decomposition where the coefficients associated with the $LL_2$ subband are also grouped into $2\times 2$ blocks but one fourth of them (marked with a *) does not have offsprings.

<img src="spatial-orientation-tree.png" alt="Example of spatial orientation tree" width="300"/>

Spatial orientation trees are the SPIHT's counterpart of the *zero trees* introduced earlier for the EZW algorithm. As we shall see later, zero trees in SPIHT are denoted as *insignificant sets*. In the SPIHT method a spatial orientation tree rooted at a particular coefficient's location $(i, j)$, constitutes a zero tree. The main difference with the EZW method is that a zero tree in SPIHT does not need to have its root equal to zero for a particular bitplane $n$. Moreover, SPIHT considers an additional type of zero trees where not only the root is excluded but also its four children. The first type of zero tree (insignificant set) is denoted in the SPIHT method as **Type A**, whilst the second as **Type B**.

Considering the spatial orientation tree and coefficient location $(i, j)$, the following sets of coefficients coordinates are defined in SPIHT to present the pseudo-code for the improved version of the sorting pass:
 * $\mathcal{C}(i,j)$: set of coordinates for the children of node $(i,j)$
 * $\mathcal{D}(i,j)$: set of coordinates for all descendants of node $(i,j)$
 * $\mathcal{G}(i,j)$: set of coordinates for all grandchildren, grand grandchildren of node $(i,j)$. Accordingly, $\mathcal{G}(i,j) = \mathcal{D}(i,j) - \mathcal{C}(i,j)$.

Spatial orientation trees and associated coordinates are used by SPIHT to define the partitioning rules of the sorting pass. Specifically, the following rules are defined:
 1. The initial partitioning is formed by all coordinates of $LL$ coefficients in the highest level of the Wavelet decomposition pyramid, i.e. $\{(i,j)\} \cup \mathcal{D}(i,j), \forall (i,j)\in LL_D$
 1. If $\mathcal{D}(i,j)$ is significant for a given $n$, then it is partitioned into a new set given by its grandchildren and associated descendant plus all its four direct children. Mathematically we have: $\{\mathcal{G}(i, j) \cup \{(k,l)\}, \forall(k,l)\in \mathcal{C}(i,j)\}$
 1. If $\mathcal{G}(i,j)$ is significant for a given $n$, then it is partitioned into four sets given by the descendants of its four children. Mathematically this corresponds to the new set: $\{(k,l), \forall(k,l)\in\mathcal{C}(i,j)\}$.

### The SPIHT coding algorithm

The different sets of coefficients coordinates created and processed by SPIHT are kept into three different lists, defined as follows:
 1. List of Significant Coefficients (LSC)
 1. List of Insignificant Coefficients (LIC)
 1. List of Insignificant sets of Coefficients (LIS). Sets here can be of Type A or B as introduced earlier.

The new encoding process for SPIHT is thus defined in the following pseudo code.

1. **Initialisation**: write $n = \lfloor \log_2(\max_{i,j}\{|c_{i,j}|\})\rfloor$ in the bitstream. Set LSC to empty, LIC to all coordinates $(i,j)$ of $LL_D$ coefficients and LIS to all coordinates in LIC which have children. Mark all entries in LIS as Type A.
1. **Sorting pass**:
    1. For each entry $(i,j)$ in LIC do:
        * Write $S_n(i,j)$
        * If $S_n(i,j)=1$ then move $(i,j)$ to the LSC and write the sign of $c_{i,j}$
    1. For each entry $(i,j)$ in LIS do:
        * If the entry is of Type A then
            * Write $S_n(\mathcal{D}(i,j))$
            * If $S_n(\mathcal{D}(i,j))=1$ then
                * For each $(k,l) \in \mathcal{C}(i,j)$ do:
                    * Write $S_n(k,l)$
                    * If $S_n(k,l)=1$ then add $(k,l)$ to the LSC and output the sign of $c_{k,l}$
                    * If $S_n(k,l)=0$ then add $(k,l)$ to the end of LIC
                * If $\mathcal{G}(i,j)$ is not empty then move $(i,j)$ to the end of the LIS as entry of Type B
            * If the entry is of Type B then
                * Write $S_n(\mathcal{G}(i,j))$
                * If $S_n(\mathcal{G}(i,j))=1$ then
                    * Add each $(k,l) \in \mathcal{C}(i,j)$ to the end of LIS as Type A entry
                    * Remove $(i,j)$ from the LIS
1. **Refinement pass**: For each entry $(i,j)$ in the LSC, except those included in the last sorting pass (i.e. having the same $n$), write the $n$-th MSB of $c_{i,j}$
1. Set $n = n - 1$ and go to Step 2

In the pseudo-code above, the `for each` loop at Step 2.2 is intended to run through *all* entries of the LIS, including those which are added within the loop and at the end of the list. This consideration is important for implementation purposes. We mentioned earlier that the sorting algorithm relies on the general principle whereby the execution path of any algorithm can be completely recovered by knowing the result of the comparisons operated at its branching points. We now note that the significance bits $S_n(\cdot)$ are indeed associated with these comparisons and therefore the decoding procedure above can easily be obtained by the pseudo-code above by simply replacing each instance of *write* with *read*. Furthermore, recalling the first version of the sorting algorithm introduced, we note that the *coordinates of those transform coefficients $c_{i,j}$ such that $2^n \leq |c_{i,j}| < 2^{n+1}$* are obtained when significant coefficients are added to the LSC in the SPIHT's version of the sorting algorithm.

## Implementing SPIHT in the SWIC codec

After we introduced the SPIHT method from the conceptual point of view, it is worthwhile to describe how this technique can be implemented in the SWIC - Simple Wavelet Image Codec - compression format to add support for quality scalability. The following subsections will discuss in a greater level of detail how the Python source of the SWIC codec has been refactored to accomodate for the processing associated with SPIHT. The focus will then move to overview the main processing associated with the entropy coding method specified by SPIHT, followed by a description on how the syntax specified by the SWIC compression format needs to be modified to support quality scalability.

### Preliminary remarks
It should be noted that the SPIHT method only assumes that the Wavelet transformation coefficients are represented in integer precision but it does not prescribe on the precision adopted. Accordingly, one can think to replace the entropy coding procedure of an SWIC complaint image codec complaint with the one specified by SPIHT. In that case, the input data to the entropy coding stage, will be the transform coefficients produced by the quantisation stage, whose integer precision is given by the Quantisation Parameter (QP) selected. Quality scalability will then provide progressive image distortion reduction, eventually resembling the distortion level associated with the QP chosen. Based on this consideration, the source code of the SWIC encoder and decoder has been refactored to accomodate the insertion of SPIHT as an alternative entropy coding method.

Before delving into the implementation details, we recall that the implementation presented in this tutorial uses the Python programming language due to its powerful `numpy` library allowing for simple and compact coding of image processing tasks (e.g. frequency transformation). The main drawback here is that the code will (inevitably) be slower due to the interpret nature of Python. On this note, it is worth to point out a comprehensive and simply written C++ implementation of SPIHT by Rene' Puchinger in his [image shrinker code base](https://github.com/rene-puschinger/imshrinker). The main differences between image shrinker and the one described in this tutorial are as follows:
 * Image shrinker uses a fixed number of Wavelet decomposition levels, which is equal to $\lceil\log_2(\max(W, H)) \rceil$ where $W$ and $H$ denoted the image width and height, respectively
 * Image shrinker does not perform any quantisation (aside from converting Wavelet's coefficient in integer precision) and therefore adjusts the coding rate based on the value provided as input to the encoder

### Refactoring the SWIC source code
In the original implementation of the SWIC codec, transform coefficients are organised into a Python list whereby the $i$-th element represents the group of Wavelet subbands associated with decomposition level $i$. These subbands could be either three (i.e. $HL$, $LH$ and $HH$) or four (i.e. $LL$, $HL$, $LH$ and $HH$) in case $i$ is equal to the maximum levels of decomposition ($D$). The list carrying all subbands for all decomposition levels is then passed to the uniform quantiser module which runs through all elements and performs quantisation of all subbands. 

Since the SPIHT method requires to traverse all image coefficients multiple times to determine whether a set of coefficients is significant, this list-based data structure may not be ideal given that one needs to keep track of the coefficients' successor in the SPIHT's spatial orientation trees. A better data structure would consist in a simple 2D array where each entry is the Wavelet coefficient associated with a particular subband, decomposition level and colour plane. Considering the Mallat decomposition adopted in the SWIC format, in such a 2D array entries whose $(x, y)$ coordinates are within the range $[0, \frac{W}{2^D}]\times[0, \frac{H}{2^D}]$ would corresponds to the $LL$ subband of a given colour plane (i.e. Y, Cb or Cr). Entries whose $(x, y)$ coordinates are within the range $[\frac{W}{2^D}, \frac{W}{2^{D-1}}]\times[\frac{H}{2^D}, \frac{H}{2^{D-1}}]$ would corresponds to the $HL$ subband at decomposition level $D$ and so on. In the following, this 2D array will be denoted in the code as `image_levels`. Coefficients quantised in `image_levels` will then be passed to the module implementing SPIHT's entropy coding so that the payload for bitstream can be produced.

It should be noted that the new arrangement in `image_levels` would require the refactor also the Python functions implementing inverse quantisation and inverse Wavelet transform, required by both the decoder as well as the encoder, this latter in case the user selects to output the reconstructed image. However, since these functions constitute the core of the non quality scalable SWIC codec, their API is not modified but rather in the SPIHT-based SWIC encoder and decoder quantised coefficient are re-organised to be brought back in the list-based implementation.

### Implementing the SPIHT entropy coding processing
One key element needed to the SPIHT implementation is the so-called successor map. This can be seen 3D Python array of size $W\times H\times 2$ whereby entry at $(i, j, 0)$ indicates the column value of the children of node at $(i,j)$ whilst entry at $(i, j, 1)$ indicates the row value of the children of node at $(i,j)$. The Python function which computes such a successor map is `compute_successor_map` in the [entropy_spiht](./entropy_spiht.py) package. This function will assign a -1 value in the output array in case a node does not have any successor, i.e. it's either a leaf of the spatial orientation tree or it is a root node in the $LL$ subband which was marked with a "*" in the picture above depicting spatial orientation trees.

Entropy encoding as specified in the SPIHT method is performed by the `encode_image_spiht` function which receives both the `image_levels` and the successor map 2D arrays. The function returns two monodimensional arrays containing the length in bytes associated with the bitstream of colour component and the bitstream itself. The function implements in Python the pseudo-code of SPIHT presented earlier. A few points worth noting in the implementation are as follows:
 1. Lists LSC, LIS and LIC are implemented as Python's [lists](https://docs.python.org/3/tutorial/datastructures.html) so that useful functions such as `pop` can be used to manipulate the sets of coefficients coordinates.
 1. The `for` loops at Steps 2.1 and 2.2 in the pseudo-code are implemented as Python `while` loops using integer (`i`) as control variable which terminates the loop in case its value exceeds the size of list LIC, LIS or LSC.
 1. The refinement step in the pseudo-code predicates that all coefficients in the LSC need to be refined except those added in the current iteration. In function `encode_image_spiht` this is implemented by checking whether the absolute value of a given coefficient is greater than or equal to $2^{n+1}$: in this way coefficients which became significant at iteration $n$ (and were added to the LSC) are not included in the refinement step associated with $n$.

At high level the processing performed by the `encode_image_spiht` function can be described by the following Python style pseudo-code:
```Python
for comp in range(image components):
    write n[comp] to the bitstream
    run SPIHT over all coefficients of comp
    write the bits produced by SPIHT to the bitstream
```

Two additional support Python functions are implemented and called from `encode_image_spiht`: `is_setA_significant` and `is_setB_significant`. As their names suggest, these functions check whether a given set of coefficients coordinates is significant or not. Recall the difference between sets of Type A and B explained above, these two Python functions will different in the sets of descendants searched. Function `is_setA_significant` will consider all children, grandchildren, great-grandchildren, etc. of coordinates $(i,j)$ whilst function `is_setB_significant` will start its analysis from the grandchildren of $(i, j)$. The processing proceeds first on a per component basis and then within each component, on per bitplane basis.

The Python classes implementing bit writing and reading functionalities, [BitWriter](./bit_io.py) and [BitReader](./bit_io.py) had to be extended to the following new implementations:
 * [BitWriterAppend](./bit_io.py): This is conceptually identical to [BitWriter](./bit_io.py) except for the fact the buffer used to store the bitstream has infinite length and this is implemented using a Python list where each byte is appended as soon as the 32 bit register used to write bitstream's bits needs to be transferred to the output buffer. In a practical implementation, the byte buffer can still have an *a priori* size set to the number of transform coefficients in each colour component, multiplied by the number of bits used in the coefficients' integer representation. The choice of using a growing Python list as output buffer has been dictated by the need to avoid to have large buffer to deal in an already slow implementation of the SPIHT method
 * [BitReaderLimited](./bit_io.py): Also this class is conceptually identical to [BitReader](./bit_io.py) saved for the fact that it is initialised with a maximum number of bits to read from the input buffer. As soon as the number of input bits exceeds such a limit, the class will throw an `EndOfParsing` exception, which is a derived class of the Python `Exception` class. The emission of an `EndOfParsing` exception is used to communicate to the entropy decoding module to stop its processing and move to the next stages such as entropy decoding of the next colour component (if any), reconstruction and inverse Wavelet transform.

Entropy decoding is implemented in the `decode_image_spiht` Python function from the [entropy_spiht](./entropy_spiht.py) package. The main processing performed by this function can be summarised by the following pseudo-code:

```Python
for comp in range(image components):
    read n[comp]
    run SPIHT where you each write instance has been replaced by read
    store the reconstructed transform coefficient levels to the output image array
```

As mentioned earlier, the SPIHT and decoding process are pretty simmetrical, hence the same helper functions used during encoding (i.e. `compute_successor_map`, `is_setA_significant` and `is_setB_significant`) are also used for decoding. Execution control via the `EndOfParsing` exception is implemented using a `try`/`catch` block inside `decode_image_spiht`. If an instance of the `BitReaderLimited` class throws an `EndOfParsing` exception, parsing stops and in the `catch` block transform coefficients are first assigned to their sign and then stored in the output coefficient array. The parsing then continues over the next colour component (if any). The maximum number bits to read given to any instance of `BitReaderLimited` is either given by the user, via command line parameter `--rate,-r` in bits per pixel units, or it is set to: $W\times H\times 16 + 4$ for each colour component. The additional four added int the previous formula is due to the number of bits used to signal $n$.Besides specifying the maximum number of bits per pixel $bpp$, the user can also select the amount of coding bits to be parsed for the luma and chroma components via command line parameter `-wy,--weighty` which sets weight $w_Y$ for the luma component. Accordingly, the bit share distribution between luma and chroma is given as:

$$
bpp_Y = w_Y\times bpp \times W\times H\\
bpp_{Cb|Cr} = \frac{(1 - w_Y)}{2}\times bpp \times W\times H
$$

Accordingly, the `BitReaderLimited` instance for each colour component will receive the above maximum number of bits to decode.

### Extending the high level syntax of the SWIC image compression format
The syntax of the SWIC format needs to be modified to accomodate the fact that entropy coding is now achieved by using the SPIHT method. The Python style pseudo-code presented in the previous subsection has also highlighted a key difference with respect to the original SWIC format. In fact, in the SPIHT method the bitstream is formed by the entropy bits associated with the luma (Y) component, followed by those of Cb and Cr components. This contrasts with the concept of *code block* introduced in SWIC to enable parallel execution of entropy encoding and decoding. Thanks to the use of code blocks, an independently decodable bitstream is produced by interleaving all entropy bits associated with all colour components of each code block. The following picture depicts the main difference between the bitstream produced by the original SWIC format and SPIHT.

<img src="bitstream-comparison.png" alt="Comparison of the bitstream generated by SWIC and SPIHT" width="600"/>

As may be noted, SWIC allows to run an entropy encoding and decoding process for each code block, making these processes parallelisable. Conversely, the bitstream produced by SPIHT doesn't allow for such a fine grain parallelism. To enable a minimal level of parallel encoding and decoding, the bitstream is modified by inserting headers signalling the size (in byte units) of each colour component's bitstream. Such header field has four byte size as depicted in the following picture.

<img src="spiht-bitstream.png" alt="SPIHT bitstream with size headers and byte stuffing" width="800"/>

The picture above also shows an initial header denoted as *Byte stuffing* with variable length set to $N$. These initial bytes are inserted in the implementation of SPIHT proposed in this tutorial to make a fair rate-distortion comparison with the SWIC codec. In fact, as mentioned in this [tutorial](./image-coding-wavelet.ipynb), each code block requires a two byte header field to be inserted so that parallel decoding (and encoding) can be achieved. Given that the SPIHT's implementation proposed in this tutorial only allows for colour plane-based parallelism, the two byte headers are not inserted, making the coding rate provided by SWIC with the SPIHT method smaller for the same level of PSNR, hence leading to a non apple-to-apple comparison. Accordingly, in the implementation proposed for the SPIHT method, the number of code blocks ($N_{CB}$) associated with the current image are calculated and the following number of zero value bytes is inserted: $N_{CB}\times 2 - 4$. Minus four is because the length of the byte stuffing chunk is signalled using four bytes.

## Coding efficiency assessment
We are in a position to compare the coding efficiency of SWIC with the SPIHT method. There two angles along which the appraisal of SPIHT could develop. The first angle appreciates the coding efficiency of SPIHT compared with the baseline SWIC format; being SPIHT ultimately another entropy coding method, it is interesting to understand how much compression ratio is gained for the additional complexity (due to bitplan encoding) associated with SPIHT. The second angle consists in picturing the quality scalability support offered by SPIHT, most notably in providing progressive quality improvement.

The following Python code cell defines and implements some support code to launch both the SWIC and SWIC+SPIHT encoders/decoders so that their coding rate and image quality can be measured. As usual, image quality in reported in terms of the Peak-Signal-to-Noise-Ratio (PSNR) across the three colour planes (i.e. Y, Cb, Cr) whilst coding rate is indicated as bits per pixel [bpp].

In [ ]:
# This Python code defines functions to gather rate-distortion data for both the SWIC and SWIC+SPIHT codecs.

import numpy as np
from encoder_spiht import swic_encoder_spiht
from decoder_spiht import swic_decoder_spiht
from encoder import swic_encoder
from decoder import swic_decoder
from pathlib import Path
import re
from dwt import DwtType
import os
from numpy.typing import NDArray
from typing import Tuple
from time import time


def get_rd_data_spiht(image_ycbcr: NDArray[np.int32],
                      qps: NDArray[np.int32],
                      dwt_levels: int,
                      dec_bpp: float = None,
                      weightl: float = 0.5,
                      output_dir: str = "") -> Tuple[NDArray[np.float64], NDArray[np.float64]]:
    if len(image_ycbcr.shape) == 3:
        rows, cols, components = image_ycbcr.shape
    else:
        components = 1
        rows, cols = image_ycbcr.shape
    rate = np.zeros(len(qps))
    psnr_total = np.zeros((len(qps), components))
    for idx_qp, qp in enumerate(qps):
        output_bitstream = f"{output_dir}/bitstream_qp{qp}.spiht"
        # Encoding
        coding_bytes, _ = swic_encoder_spiht(image_ycbcr, output_bitstream, qp, 8, dwt_levels, DwtType.CDF9_7, False)

        # Check coding bytes
        actual_bytes = os.stat(output_bitstream).st_size
        assert actual_bytes == coding_bytes

        # Coding rate
        rate[idx_qp] = coding_bytes * 8 / rows / cols

        # Decoding
        image_decoded = swic_decoder_spiht(output_bitstream, 0, dec_bpp, weightl, False)

        # Compute the PSNR
        if components == 1:
            mse = np.mean(np.square(image_ycbcr[:, :] - image_decoded[:, :].astype(np.int32)))
            psnr_total[idx_qp, 0] = 10 * np.log10(255**2 / mse)
        else:
            for comp in range(components):
                mse = np.mean(np.square(image_ycbcr[:, :, comp] - image_decoded[:, :, comp].astype(np.int32)))
                psnr_total[idx_qp, comp] = 10 * np.log10(255**2 / mse)
    return rate, psnr_total

def get_rd_data_swic(image_ycbcr: NDArray[np.int32],
                     qps: NDArray[np.int32],
                     dwt_levels: int,
                     output_dir: str = "") -> Tuple[NDArray[np.float64], NDArray[np.float64]]:
    if len(image_ycbcr.shape) == 3:
        rows, cols, components = image_ycbcr.shape
    else:
        components = 1
        rows, cols = image_ycbcr.shape
    rate = np.zeros(len(qps))
    psnr_total = np.zeros((len(qps), components))
    for idx_qp, qp in enumerate(qps):
        output_bitstream = f"{output_dir}/bitstream_qp{qp}.swic"
        # Encoding
        coding_bytes, _ = swic_encoder(image_ycbcr, output_bitstream, qp, 8, dwt_levels, DwtType.CDF9_7, False)

        # Check coding bytes
        actual_bytes = os.stat(output_bitstream).st_size
        assert actual_bytes == coding_bytes

        # Coding rate
        rate[idx_qp] = coding_bytes * 8 / rows / cols

        # Decoding
        image_decoded = swic_decoder(output_bitstream, 0, False)

        # Compute the PSNR
        if components == 1:
            mse = np.mean(np.square(image_ycbcr[:, :] - image_decoded[:, :].astype(np.int32)))
            psnr_total[idx_qp, 0] = 10 * np.log10(255**2 / mse)
        else:
            for comp in range(components):
                mse = np.mean(np.square(image_ycbcr[:, :, comp] - image_decoded[:, :, comp].astype(np.int32)))
                psnr_total[idx_qp, comp] = 10 * np.log10(255**2 / mse)
    return rate, psnr_total

### Rate-distortion performance of SPIHT integrated in the SWIC codec
In this section we measure the coding efficiency of SPIHT when integrated in the SWIC codec over the Common Intermediate Format (CIF) `foreman` test image. The following Python code cell defines a workflow where the test image is read into a `numpy` 3D array and then passed through both the helper functions defined above to gather the rate-distortion data for both codes. The user can add more `.yuv` test images to the `test_images` Python list below so that more test data can be generated. The workflow stores bitstreams and rate-distortion data for both codecs in the `experimental_data` directory. Such a directory is cleared by any old bitstream file before the simulation runs.

In [ ]:
from pathlib import Path

test_images = ["../../input-data/foreman_352x288_30Hz_8b_P444.yuv"]
test_qps = np.array([22, 27, 32, 37], np.int32)
data_dir = "experimental-data"

# Wipe out all experimental data created from previous runs
working_dir = Path(f"{os.getcwd()}/{data_dir}")
files_spiht = working_dir.glob("*.spiht")
files_swic = working_dir.glob("*.swic")

for f in files_spiht:
    os.remove(f)
for f in files_swic:
    os.remove(f)

levels, components = 5, None
complexity_swic = np.zeros(len(test_images))
complexity_spiht = np.zeros(len(test_images))

for idx_image, image in enumerate(test_images):
    extension_type = Path(image).suffix
    # Read image
    if extension_type == ".yuv":
        components = 3
        pattern_size = r"(\d+)x(\d+)"
        matches = re.search(pattern_size, image)
        cols, rows = matches.groups()
        rows, cols = int(rows), int(cols)
        image_ycbcr = np.zeros((rows, cols, 3), np.int32)
        with open(image, "rb") as fh:
            for comp in range(components):
                image_ycbcr[:, :, comp] = np.reshape(np.frombuffer(fh.read(rows * cols), dtype=np.uint8), (rows, cols))
    else:
        raise NotImplemented

    start = time()
    rate_spiht, psnr_spiht = get_rd_data_spiht(image_ycbcr, test_qps, levels, None, 0.5, data_dir)
    stop = time()
    complexity_spiht[idx_image] = stop - start
    start = time()
    rate_swic, psnr_swic = get_rd_data_swic(image_ycbcr, test_qps, levels, data_dir)
    stop = time()
    complexity_swic[idx_image] = stop - start
    # Save metrics arrays
    np.save(f"{data_dir}/rate_spiht_img{idx_image}.npy", rate_spiht)
    np.save(f"{data_dir}/psnr_spiht_img{idx_image}.npy", psnr_spiht)
    np.save(f"{data_dir}/rate_swic_img{idx_image}.npy", rate_swic)
    np.save(f"{data_dir}/psnr_swic_img{idx_image}.npy", psnr_swic)

Now that the rate-distortion data have been generated, a plot can be produced to compare the curves for both codecs. We also include the usual Bjontegaard Delta (BD) on Rate (BDR) to quantify any coding gain along with the run time spent by both codecs in processing the four Quantisation Parameter (QP) values. The following Python cell implements the plot generation part considering the PSNR over the luma component as the quality measure.

In [ ]:
import matplotlib.pyplot as plt

def bd_rate(ra: NDArray[np.float64], rt: NDArray[np.float64], da: NDArray[np.float64], dt: NDArray[np.float64]) -> float:
    log_ra = np.log(ra)
    log_rt = np.log(rt)

    pa = np.polyfit(da, log_ra, 3)
    pt = np.polyfit(dt, log_rt, 3)

    min_int = max(np.min(da), np.min(dt))
    max_int = min(np.max(da), np.max(dt))

    pa_int = np.polyint(pa)
    pt_int = np.polyint(pt)

    ia = np.polyval(pa_int, max_int) - np.polyval(pa_int, min_int)
    it = np.polyval(pt_int, max_int) - np.polyval(pt_int, min_int)

    diff_e = (it - ia) / (max_int - min_int)
    diff = (np.exp(diff_e) - 1) * 100

    return diff

# Hard coded assumption: the Foreman image is always identified by index 0
del rate_spiht, rate_swic, psnr_spiht, psnr_swic
rate_spiht = np.load(f"{data_dir}/rate_spiht_img0.npy")
rate_swic = np.load(f"{data_dir}/rate_swic_img0.npy")
psnr_spiht = np.load(f"{data_dir}/psnr_spiht_img0.npy")
psnr_swic = np.load(f"{data_dir}/psnr_swic_img0.npy")

bdr = bd_rate(rate_swic, rate_spiht, psnr_swic[:, 0], psnr_swic[:, 0])

plt.figure(figsize=(8, 6))
plt.plot(rate_swic, psnr_swic[:, 0], "r-*", linewidth=2, label=f"swic, proc. time {complexity_swic[0]:.1f} [s]")
plt.plot(rate_spiht, psnr_spiht[:, 0], "b-o", linewidth=2, label=f"spiht, proc. time {complexity_spiht[0]:.1f} [s]")
plt.grid()
plt.xlabel("Rate [bpp]", fontsize=16)
plt.ylabel("PSNR-Y [dB]", fontsize=16)
plt.title(f"Foreman CIF, BDR = {bdr:.1f}%", fontsize=16)
plt.legend();

As may be noted from the plot above, the SPIHT method provides a superior coding efficiency which is purely associated with its entropy encoding algorithm (remember that both codecs use the same quantised transform coefficients). Such a better coding efficiency is quantified by an average coding rate reduction above 22% for the test QPs used earlier. The reader may wonder what is the increase in encoding complexity to be paid for this better coding efficiency. Although we are dealing with a Python implementation which is neither multithreaded, nor optimised, a tenfold processing time increase has been observed on the author's machine for the SWIC codec which uses SPIHT as entropy encoding module. Such a factor is particularly experienced at low QP values where more coefficients need to be entropy encoded. Put simply, to achieve fine grain quality scalability, SPIHT performs bitplane encoding which would, in principle, require an increase of a factor $n$ (i.e. number of bitplanes) in the number of operations required for each transform coefficient during entropy encoding.

### Quality scalability support offered by SPIHT
The next dimension to appraise for the SPIHT method is its ability to support progressive quality improvement so to trade variable bandwidth or decoding complexity for image quality. The following Python cell defines a function (`decode_spiht_bpp`) which calls the SWIC's decoder using the SPIHT entropy decoding method. The function receives as inputs the bitstream to be decoded, the original image organised as 3D `numpy` array, the number of Wavelet levels to reconstruct, along with the bits per pixel to parse. This latter parameter is increased throughout the experiment in order to increase the quality (PSNR) of the reconstructed image. Two calls of `decode_spiht_bpp` are made with different weighting of coding bits to be decoded for the luma component. We mentioned earlier that the `decode_image_spiht` receives as input a weighting factor which regulates the share of bits parsed from the input will receive the luma component.

In [ ]:
def decode_spiht_bpp(input_bitstream: str,
                     image_ycbcr: NDArray[np.int32],
                     dwt_levels: int,
                     test_bpps: NDArray[np.float64],
                     weightl: float) -> Tuple[NDArray[np.float64], NDArray[np.float64]]:
    if not os.path.exists(input_bitstream):
        raise Exception(f"Input bitstream: {input_bitstream} doesn't exist")

    if len(image_ycbcr.shape) == 3:
        components = image_ycbcr.shape[2]
    else:
        components = 1

    psnr_total, decoding_time = np.zeros((len(test_bpps), components)), np.zeros(len(test_bpps))
    for idx, bpp in enumerate(test_bpps):
        # Decoding
        start = time()
        image_decoded = swic_decoder_spiht(input_bitstream, dwt_levels, bpp, weightl, False)
        stop = time()
        decoding_time[idx] = stop - start

        # Compute the PSNR
        if components == 1:
            mse = np.mean(np.square(image_ycbcr[:, :] - image_decoded[:, :].astype(np.int32)))
            psnr_total[idx, 0] = 10 * np.log10(255**2 / mse)
        else:
            for comp in range(components):
                mse = np.mean(np.square(image_ycbcr[:, :, comp] - image_decoded[:, :, comp].astype(np.int32)))
                psnr_total[idx, comp] = 10 * np.log10(255**2 / mse)

    return decoding_time, psnr_total

input_bitstream = "experimental-data/bitstream_qp27.spiht"
bpps = [.8, 1, 1.2, 1.4, 1.6, 2.2, 3.25]
levels = 0
# This is hard coded
idx_qp27 = 1

# Measure decoding time and PSNR for the test bpp values
decoding_time, decoding_psnr = decode_spiht_bpp(input_bitstream, image_ycbcr, levels, bpps, 0.5)
decoding_time_w, decoding_psnr_w = decode_spiht_bpp(input_bitstream, image_ycbcr, levels, bpps, 0.72)

The PSNR values and decoding times collected when running the decoder are now organised in two `matplotlib` graphs by the following Python code cell. From the plots obtained, we can observe three main aspects:
 1. The weighting factor for the luma component plays a key role in determining how fast the decoded image quality will converge to the PSNR value of the fully decoded bitstream.
 1. The PSNR curve associated with weighting equal to 0.72 presents a step gradient meaning that the SPIHT algorithm efficiently organises the coding bits in such a way that with little investement in decoding complexity (and/or transmission bits) a good share of the compressed original quality can be recovered.
 1. Decoding time is also influenced by the weighting factor for the luma component, being this colour plane the one consuming a large part of the decoder's complexity.

In [ ]:
# Plot the results
psnr_full = psnr_spiht[idx_qp27, 0] * np.ones(decoding_psnr.shape[0])
total_time = decoding_time_w[-1] * np.ones(len(bpps))

fig, (ax1, ax2) = plt.subplots(1, 2)
fig.set_figwidth(20)
fig.set_figheight(8)
ax1.plot(bpps, psnr_full, "k-", linewidth=2, label="PSNR-Y full decoding")
ax1.plot(bpps, decoding_psnr[:, 0], "g-*", linewidth=2, label="Progressive decoding, $w_l$ = 0.5")
ax1.plot(bpps, decoding_psnr_w[:, 0], "m-*", linewidth=2, label="Progressive decoding, $w_l$ = 0.72")
ax1.grid()
ax1.set_xlabel("Number of bits per pixel decoded", fontsize=16)
ax1.set_ylabel("PSNR-Y [dB]", fontsize=16)
ax1.legend();

ax2.plot(bpps, total_time, "k-", linewidth=2, label="Full parsing")
ax2.plot(bpps, decoding_time, "g-*", linewidth=2, label="Progressive decoding, $w_l$ = 0.5")
ax2.plot(bpps, decoding_time_w, "m-*", linewidth=2, label="Progressive decoding, $w_l$ = 0.72")
ax2.grid()
ax2.set_xlabel("Number of bits per pixel decoded", fontsize=16)
ax2.set_ylabel("Decoding time [s]", fontsize=16)
ax2.legend()
fig.suptitle("Foreman CIF", fontsize=16);

## Conclusions

This tutorial has explored image coding techniques to produce a progressive quality and scalable bitstream. Quality scalability is obtained via bitplane encoding techniques whereby the bits associated with each transform coefficient bitplane are written to make the bitstream embedded, so to guarantee a progressive decoded quality refinement. Among the different bitplane encoding techniques proposed for Wavelet-based image compression, this tutorial reviewed in a greater level of detail the SPIHT method due to its high coding efficiency and relative simplicity to implement. SPIHT has been used as drop in replacement for the entropy coding procedure of the SWIC format and then the rate-distortion performance has been compared to the SWIC original design. A significantly improved coding efficiency has been presented due to the efficient use of SPIHT of zero tree relationships associated with the Wavelet transform decomposition. Support for quality scalability has also been assessed to show how well SPIHT organises the coding bitplanes bits to allow a progressive and graceful decoding quality improvement. Improved coding efficiency and support for quality scalability come at the price of significant increase in the entropy encoding and decoding algorithms. Moreover, the ability to perform parallel processing is diminished by the design of SPIHT which requires access to all coefficients in different subbands. Although parallel processing can still be applied at colour plane and region level (e.g. using the same concept of tiles as specified in the JPEG 2000), coding block level parallelism as offered by the original SWIC design would only be possible if a different bitplane encoding method (e.g. EBCOT) is employed.